# BCI for Inner Speech Decoding: Preprocessing and Feature Extraction

In [ ]:
import sys
sys.path.append(".")

from utils import bandpass, remove_artifacts, baseline_correction, apply_notch_filter, bandpass

## Preprocessing

In [ ]:
# Extract only the relevant time window from each trial
# X goes from [trials × channels × full_time_points] to [trials × channels × selected_time_points]
X = select_time_window(X=X, t_start=t_start, t_end=t_end, fs=fs)

print("Time window selection complete!")
print("\nNew X (EEG data) shape:", X.shape)
print("  - Number of trials:", X.shape[0])
print("  - Number of EEG channels:", X.shape[1])
print("  - Number of time points per trial:", X.shape[2])
print("  - Selected duration per trial:", X.shape[2] / fs, "seconds")

# Calculate how many data points were selected
total_data_points = X.shape[0] * X.shape[1] * X.shape[2]
print(f"\nTotal data points in selected window: {total_data_points:,}")

In [ ]:
# First, let's reload the original data for our subject
X, Y = extract_data_from_subject(root_dir, N_S, datatype)

# Extract only the relevant time window
X = select_time_window(X=X, t_start=t_start, t_end=t_end, fs=fs)

# Define which conditions and classes we want to compare
# We'll compare "Up" vs "Down" in the "Inner Speech" condition
Conditions = [["Inner"], ["Inner"]]  # All groups use Inner Speech condition
Classes = [["Up"], ["Down"]]  # First group is "Up", second is "Down"

print("Filtering data to compare:")
print(f"- Group 1: {Classes[0][0]} direction in {Conditions[0][0]} condition")
print(f"- Group 2: {Classes[1][0]} direction in {Conditions[1][0]} condition")
# print(f"- Group 3: {Classes[2][0]} direction in {Conditions[2][0]} condition")
# print(f"- Group 4: {Classes[3][0]} direction in {Conditions[3][0]} condition")

# Transform and filter the data to keep only trials of interest
X, Y = transform_for_classificator(X, Y, Classes, Conditions)



print("\nAfter filtering:")
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("\nY now contains simple binary labels:")
print("- 0 represents:", Classes[0][0])
print("- 1 represents:", Classes[1][0])

# Check how many trials we have for each class
n_up = np.sum(Y == 0)
n_down = np.sum(Y == 1)
# n_left = np.sum(Y == 2)
# n_right = np.sum(Y == 3)

print(f"\nNumber of '{Classes[0][0]}' trials: {n_up}")
print(f"Number of '{Classes[1][0]}' trials: {n_down}")
# print(f"Number of '{Classes[2][0]}' trials: {n_left}")
# print(f"Number of '{Classes[3][0]}' trials: {n_right}")

In [ ]:
selected_channels = np.arange(128)

X = X[:, selected_channels, :]

### CSP

In [ ]:
csp = CSP(
    n_components=8,
    reg='ledoit_wolf',
    log=True,
    norm_trace=False
)

## Apply preprocessing pipeline

In [ ]:
# Step 1: Resample
X = resample_data(X, sfreq, target_sfreq)

# Step 2: Bandpass filter
X = apply_bandpass(X, lowcut, highcut, target_sfreq)

# Step 3: Notch filter
X = apply_notch_filter(X, notch_freq, target_sfreq)

# Step 4: Baseline correction
# X = baseline_correction(X)

# Step 5: Artifact removal
# X, kept_indices = remove_artifacts(X)

# Y = Y[kept_indices]

print("Preprocessing complete.")
print("Processed X shape:", X.shape)

## Test Train Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    Y,
    test_size=0.3,
    random_state= 23,
    stratify=Y
)

## Apply CSP 
Ensure there are no data leaks

In [ ]:
X_train_alpha = bandpass(X_train, 8, 12, fs=128)
X_test_alpha  = bandpass(X_test, 8, 12, fs=128)

X_train_beta = bandpass(X_train, 13, 30, fs=128)
X_test_beta  = bandpass(X_test, 13, 30, fs=128)

csp_alpha = CSP(n_components=4, log=True)
csp_beta  = CSP(n_components=4, log=True)

X_train_a = csp_alpha.fit_transform(X_train_alpha, y_train)
X_test_a  = csp_alpha.transform(X_test_alpha)

X_train_b = csp_beta.fit_transform(X_train_beta, y_train)
X_test_b  = csp_beta.transform(X_test_beta)

X_train_final = np.hstack([X_train_a, X_train_b])
X_test_final  = np.hstack([X_test_a, X_test_b])